# 08 · The model cycle: data → labels → training → evaluation → deployment

The weakest model is the detector: COCO YOLOX has no *goalkeeper / referee* class and misses small balls. The cycle
to improve it on **your** footage (same camera, same pitches):

```
 full-match runs ──► 1 export frames + pseudo-labels (YOLO format, split by time)
                         │
                         ▼
                     2 correct the labels (CVAT / Label Studio / Roboflow)  ◄── the human step, the one that matters
                         │
                         ▼
                     3 train  (ultralytics YOLO, imgsz 1280, GPU)
                         │
                         ▼
                     4 evaluate on held-out *corrected* frames: per-class P/R, ball recall, ms / frame
                         │            vs the current detector
                         ▼
                     5 deploy: config override → 06 on the same window → benchmarks.csv → full run
```

Other trainable parts, same cycle: **kit classifier** (your clicks = training data; leave-one-out in 04),
**xG model** (`scripts/train_xg.py`, `notebooks/04_xg_model_eval.ipynb`), **pitch keypoints** for the
`keypoints` homography (same export with landmarks instead of boxes).

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

In [ ]:
from football_eval.detection import build_detector
from football_eval.evaluation import detection_pr
from football_eval import accel
CLASSES = ["player", "goalkeeper", "referee", "ball"]          # YOLO class ids 0..3
DS = ROOT / "data" / "datasets" / f"{Path(ctx.video).stem}_det"
run_dir = Path(P.full_run_dir) if P.full_run_dir else None
run_dir = (ROOT / run_dir) if run_dir is not None and not run_dir.is_absolute() else run_dir
tracks_full = pd.read_csv(run_dir / "tracks.csv", low_memory=False) if run_dir is not None and (run_dir / "tracks.csv").exists() else None
print("dataset folder:", DS)
print("roles from a full run:", run_dir if tracks_full is not None else "none (all people exported as 'player')")

## 1 · Export frames + pseudo-labels
Frames every `EVERY_S` seconds inside the periods. Boxes from the current detector; the role (goalkeeper / referee)
from the full run's `tracks.csv` when the box matches a track box on that frame (IoU ≥ 0.5); staff / fans dropped.
**Split by time block** (alternating 5-minute blocks → train / val): neighbouring frames are near-duplicates, a random
split would leak and inflate the scores.

In [ ]:
EVERY_S, BLOCK_S, MAX_FRAMES = 20.0, 300.0, 40          # raise MAX_FRAMES for a real dataset (300-1000 frames)
from football_eval.tracking import iou_matrix
halves = cfg.match.halves or [[0, ctx.info.duration_s]]
times = np.concatenate([np.arange(a + 5, min(b, ctx.info.duration_s) - 5, EVERY_S) for a, b in halves])
times = times[np.linspace(0, len(times) - 1, min(MAX_FRAMES, len(times))).round().astype(int)]
detector = build_detector(cfg)
for sp in ("train", "val"):
    (DS / "images" / sp).mkdir(parents=True, exist_ok=True); (DS / "labels" / sp).mkdir(parents=True, exist_ok=True)
manifest = []
cap = cv2.VideoCapture(ctx.video)
for t in times:
    f = int(round(t * ctx.info.fps)); cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok, img = cap.read()
    if not ok: continue
    d = detector(img)
    cls = list(d.cls)
    if tracks_full is not None:
        g = tracks_full[(tracks_full.frame - f).abs() <= ctx.step]
        if len(g):
            g = g[g.frame == g.frame.iloc[(g.frame - f).abs().argmin()]]
            people = [i for i, c in enumerate(cls) if c != "ball"]
            if people:
                iou = iou_matrix(d.xyxy[people], g[["x0", "y0", "x1", "y1"]].to_numpy(float))
                for a_, i in enumerate(people):
                    j = int(iou[a_].argmax())
                    if iou[a_, j] >= 0.5:
                        cls[i] = str(g.cls.iloc[j])
    split = "train" if int(t // BLOCK_S) % 2 == 0 else "val"
    name = f"{Path(ctx.video).stem}_f{f:07d}"
    cv2.imwrite(str(DS / "images" / split / f"{name}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    h, w = img.shape[:2]; lines = []
    for (x0, y0, x1, y1), c in zip(d.xyxy, cls):
        if c not in CLASSES: continue                      # staff / spectator / steward are not labelled
        lines.append(f"{CLASSES.index(c)} {(x0 + x1) / 2 / w:.6f} {(y0 + y1) / 2 / h:.6f} {(x1 - x0) / w:.6f} {(y1 - y0) / h:.6f}")
    (DS / "labels" / split / f"{name}.txt").write_text("\n".join(lines), encoding="utf-8")
    manifest.append(dict(name=name, split=split, frame=f, t=round(t, 1), n_boxes=len(lines), **{c: cls.count(c) for c in CLASSES}))
cap.release()
(DS / "data.yaml").write_text(f"path: {DS.as_posix()}\ntrain: images/train\nval: images/val\nnames: {dict(enumerate(CLASSES))}\n", encoding="utf-8")
man = pd.DataFrame(manifest); man.to_csv(DS / "manifest.csv", index=False)
man.groupby("split")[["n_boxes"] + CLASSES].sum().assign(frames=man.groupby("split").size())

## 2 · Correct the labels (outside the notebook)
The pseudo-labels carry the current detector's mistakes; training on them uncorrected only teaches the model to copy
itself. Import `data/datasets/<name>_det/` as a **YOLO** dataset in CVAT, Label Studio or Roboflow, fix the boxes
(missed far players, the ball, goalkeepers / referees), export back as YOLO into the same folder. Mark progress in
`manifest.csv` (`checked` column) — the evaluation below only trusts checked frames for the val split.

In [ ]:
if "checked" not in man:
    man["checked"] = False; man.to_csv(DS / "manifest.csv", index=False)
man = pd.read_csv(DS / "manifest.csv")
print(f"checked frames: train {man[(man.split == 'train')].checked.sum()}/{(man.split == 'train').sum()}, "
      f"val {man[(man.split == 'val')].checked.sum()}/{(man.split == 'val').sum()}")

## 3 · Train
Needs `pip install ultralytics` (and the CUDA build of PyTorch on the NVIDIA machine). Set `TRAIN = True`. Small objects: train at `imgsz=1280` (or on 1024-px tiles), keep the mosaic augmentation, start from `yolo11s.pt`; ~50 epochs on a few hundred frames is a useful first model.

In [ ]:
TRAIN = False
BASE, EPOCHS, IMGSZ = "yolo11s.pt", 50, 1280
dev = accel.torch_device("auto")
cmd = f'yolo detect train data="{DS / "data.yaml"}" model={BASE} imgsz={IMGSZ} epochs={EPOCHS} batch=-1 device={dev.replace("cuda:", "")} project="{ROOT / "models" / "runs"}" name=det_{Path(ctx.video).stem}'
print("device:", dev, "\nequivalent CLI:\n ", cmd)
if TRAIN:
    from ultralytics import YOLO
    model = YOLO(BASE)
    r = model.train(data=str(DS / "data.yaml"), imgsz=IMGSZ, epochs=EPOCHS, batch=-1, device=dev,
                    project=str(ROOT / "models" / "runs"), name=f"det_{Path(ctx.video).stem}")
    print("best weights:", Path(r.save_dir) / "weights" / "best.pt")

## 4 · Evaluate: current detector vs candidate on the val split
Per class precision / recall at IoU 0.5 against the labels, ball recall within 10 px, ms per frame. Until the val labels are corrected the baseline is scored against its own output (≈ 100 %) — only the *checked* frames give a real number.

In [ ]:
CANDIDATE = None          # e.g. ROOT / "models/runs/det_<video>/weights/best.pt"
def load_labels(p, w, h):
    out = []
    for ln in p.read_text(encoding="utf-8").splitlines():
        c, cx, cy, bw, bh = ln.split(); cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        out.append((CLASSES[int(c)], [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2]))
    return out
def evaluate(det_fn, rows):
    acc, ms = {c: dict(tp=0, fp=0, fn=0) for c in CLASSES}, []
    ball_hit = ball_n = 0
    for r in rows.itertuples():
        img = cv2.imread(str(DS / "images" / r.split / f"{r.name}.jpg")); h, w = img.shape[:2]
        gt = load_labels(DS / "labels" / r.split / f"{r.name}.txt", w, h)
        t0 = time.perf_counter(); d = det_fn(img); ms.append((time.perf_counter() - t0) * 1000)
        for c in CLASSES:
            G = np.array([b for k, b in gt if k == c], float).reshape(-1, 4); Pd = d.of(c).xyxy
            if c == "ball":
                for gb in G:
                    ball_n += 1; gc = (gb[:2] + gb[2:]) / 2
                    pc = (Pd[:, :2] + Pd[:, 2:]) / 2 if len(Pd) else np.zeros((0, 2))
                    ball_hit += bool(len(pc) and np.hypot(*(pc - gc).T).min() <= 10)
            s = detection_pr(Pd, G, 0.5)
            for k in ("tp", "fp", "fn"): acc[c][k] += s[k]
    tab = pd.DataFrame(acc).T
    tab["precision"] = tab.tp / (tab.tp + tab.fp).clip(lower=1); tab["recall"] = tab.tp / (tab.tp + tab.fn).clip(lower=1)
    tab.loc["ball", "recall_10px"] = ball_hit / max(ball_n, 1)
    tab["ms_per_frame"] = np.mean(ms)
    return tab
val = man[man.split == "val"]
val_checked = val[val.checked] if val.checked.any() else val
print(f"scoring on {len(val_checked)} val frames ({'checked' if val.checked.any() else 'UNCHECKED pseudo-labels'})")
scores = {"current (" + cfg.detection.backend + ")": evaluate(detector, val_checked)}
if CANDIDATE and Path(CANDIDATE).exists():
    from football_eval.detection import UltralyticsDetector
    cand = UltralyticsDetector(str(CANDIDATE), {c: c for c in CLASSES}, cfg.detection.conf_player, cfg.detection.conf_ball, 1280, "auto")
    scores["candidate"] = evaluate(cand, val_checked)
pd.concat(scores, names=["model", "class"]).round(3)

## 5 · Deploy
A config override that switches the backend; run notebook 06 with it (`config:` in `params.yaml`) to log the KPIs next to the baseline in `benchmarks.csv`, then a full `run_analysis.py --config ...`.

In [ ]:
override = ROOT / "configs" / "candidate_detector.yaml"
if CANDIDATE:
    import yaml
    base = yaml.safe_load(open(ROOT / P.config, encoding="utf-8")) or {}
    base.setdefault("detection", {}).update(backend="ultralytics", model_path=str(Path(CANDIDATE).relative_to(ROOT)) if Path(CANDIDATE).is_relative_to(ROOT) else str(CANDIDATE),
                                            class_map={c: c for c in CLASSES}, imgsz=1280)
    override.write_text(yaml.safe_dump(base, sort_keys=False), encoding="utf-8")
    print("written", override, "\nnext: params.yaml -> config: configs/candidate_detector.yaml, run 06, compare in benchmarks.csv")
    print(f'full run: python run_analysis.py --video "{P.video}" --config configs/candidate_detector.yaml --out outputs/{Path(ctx.video).stem}_candidate')
else:
    print("set CANDIDATE to the trained weights to write configs/candidate_detector.yaml")